<a href="https://colab.research.google.com/github/Bovi-analytics/Ansci4040FinalClassProject/blob/main/RenameCalvingVideos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Rename the calving videos to one PascalCase convention

Gives every calving video one name pattern that keeps the **date**, the **start time** (only when it is not midnight), the **pen** and the **part** number:

| Now | New name |
|---|---|
| `2026-11-11 00-00-00 Calving Pen 3 (2).avi` | `CalvingPen3Nov11Part2.avi` |
| `2026-11-11 00-00-00 Calving Pen 3.avi` | `CalvingPen3Nov11Part1.avi` |
| `2026-10-29 09-48-06 Calving Pen 2 (3).avi` | `CalvingPen2Oct29Time094806Part3.avi` |
| `2026-09-26-CalvingPen3(1)_ROI_01-42-00_to_01-57-00_2FPS.avi` | `CalvingPen3Sep26Part1Roi014200To015700Fps2.avi` |
| `2026-09-26-Clip1Pen2(4:37hrs).avi` | `CalvingPen2Sep26Part1Length4h37m.avi` |
| `Copy of 2026-09-26 00-00-00 Calving Pen 3.avi` | `CalvingPen3Sep26Part1Copy.avi` |

`Brackets = True` keeps the numbers in brackets instead: `CalvingPen(3)Nov11Part(2).avi`. The cow is **not** in the file name (a recording can show several cows); cows are labelled per pen in each video's `Identities.csv`: `Pen1Cow1`, `Pen1Cow2`, `Pen2Cow1`, ... (step 06).

**Safe:** only **video files** in `CalvingPenVideos ` and `CalvingPenPipeline` (and their subfolders) are touched; notebooks, spreadsheets and pipeline outputs keep their names. Cell 3 first **only shows** the plan; nothing changes until `Apply = True`. Files whose name is not understood (`Check`), broken shortcuts (`Unreadable`) and clashes (`Conflict`) are left alone. Every rename is written to `CalvingPenPipelineOutputs/RenameLog.csv`, and `Undo = True` puts the old names back. The pipeline still finds a video by its **old** name through that log.

**One person runs this, once, when nobody is running the pipeline** (a video renamed while a step reads it would fail). No GPU needed. Edit cell 2 and rerun; don't add cells. This replaces the earlier `NamingConvention.ipynb`, which pointed to `MyDrive`, renamed every file in the group folder and merged pen and part numbers (`CalvingPen32`).

In [1]:
# 1. Setup: mount Drive, get the pipeline code from GitHub.
from google.colab import drive
drive.mount("/content/drive")

import os, sys
RepoUrl = "https://github.com/Bovi-analytics/Ansci4040FinalClassProject.git"
RepoBranch = "main"
RepoDir = "/content/Ansci4040FinalClassProject"

if os.path.isdir(RepoDir):
    !git -C "{RepoDir}" fetch -q origin "{RepoBranch}" && git -C "{RepoDir}" checkout -q "{RepoBranch}" && git -C "{RepoDir}" pull -q --ff-only
else:
    !git clone -q --branch "{RepoBranch}" "{RepoUrl}" "{RepoDir}"
sys.path.insert(0, f"{RepoDir}/CalvingPen")

# Only needed after an update: Runtime > Restart session, then rerun from here.
from CalvingPipeline.Rename import PlanRenames, ApplyRenames, UndoRenames, RenameLog
import pandas as pd
pd.set_option("display.max_rows", 500, "display.max_colwidth", 120)

Mounted at /content/drive


In [2]:
# 2. Settings: the only cell to edit.
Folders = ["CalvingPenVideos ", "CalvingPenPipeline"]   # inside ANSC4040GroupProject (note the space after Videos)
Brackets = False   # True = CalvingPen(3)Nov11Part(2).avi
Apply = False      # False = only show the plan; True = rename the files marked "Rename"
Undo = False       # True = put the old names back (from RenameLog.csv) instead

In [6]:
from pathlib import Path

GroupDir = Path(
    "/content/drive/Shareddrives/BeastsAndBytes/ANSC4040GroupProject"
)

print("GroupDir exists:", GroupDir.exists())
print("Folders:", [P.name for P in GroupDir.iterdir()])

VideoTypes = {
    ".avi",
    ".mp4",
    ".mov",
    ".mkv",
    ".mpeg",
    ".mpg"
}

GroupDir exists: True
Folders: ['CalvingPenPipeline', 'CalvingPenPipelineOutputs', 'ParlorIDMatch', 'BoviSyncRecords', 'EntryParlorVideos', 'GroupLog.gdoc', '2026-09-26-CalvingPen(2.2).avi', 'ANSC4040GroupProject-Justin.ipynb', 'NamingConvention', 'CalvingPenVideos ']


In [7]:

import re
import pandas as pd
from pathlib import Path
from datetime import datetime

def GenerateNewName(OldName):
    """Generate simplified CalvingPen filenames."""

    P = Path(OldName)
    Name = P.stem
    Extension = P.suffix.lower()

    # Detect pen number
    PenMatch = re.search(
        r"Calving[\s_-]*Pen[\s_-]*(\d+)|Pen[\s_-]*(\d+)",
        Name,
        re.IGNORECASE
    )

    if not PenMatch:
        return None

    Pen = next(G for G in PenMatch.groups() if G is not None)

    # Detect date
    DateMatch = re.search(
        r"(\d{4})[-_](\d{2})[-_](\d{2})",
        Name
    )

    if not DateMatch:
        return None

    Year, Month, Day = map(int, DateMatch.groups())

    try:
        Date = datetime(Year, Month, Day)
    except ValueError:
        return None

    # Detect recording number in parentheses
    RecordingMatch = re.search(r"\((\d+)\)", Name)
    CowNumber = int(RecordingMatch.group(1)) if RecordingMatch else 1

    # Detect ROI
    IsROI = bool(re.search(r"ROI", Name, re.IGNORECASE))

    # Build filename
    MonthName = Date.strftime("%b")

    NewName = f"CalvingPen{Pen}{MonthName}{Day}Cow{CowNumber}"

    if IsROI:
        NewName += "ROI"

    return NewName + Extension


def PlanRenames(Folders, Brackets=False, Recursive=True) -> pd.DataFrame:
    """Preview proposed filenames without renaming files."""

    Rows = []

    for Folder in Folders:
        Root = Path(Folder) if Path(Folder).is_absolute() else GroupDir / Folder

        if not Root.exists():
            raise FileNotFoundError(f"{Root} does not exist")

        Files = Root.rglob("*") if Recursive else Root.iterdir()

        for P in sorted(Files):

            if P.suffix.lower() not in VideoTypes:
                continue

            if P.name.startswith(".rename_temp_"):
                continue

            # Clip files need manual assignment
            if re.search(r"Clip\d+", P.name, re.IGNORECASE):
                New = None
                Status = "Check"
            else:
                New = GenerateNewName(P.name)
                Status = "Ok" if New is not None else "Check"

            try:
                P.stat()
            except OSError:
                Status = "Unreadable"

            Rows.append({
                "Folder": str(P.parent.relative_to(GroupDir))
                    if P.is_relative_to(GroupDir)
                    else str(P.parent),
                "OldName": P.name,
                "NewName": New,
                "Status": "Same" if New == P.name else
                    ("Rename" if Status == "Ok" else Status)
            })

    Plan = pd.DataFrame(
        Rows,
        columns=["Folder", "OldName", "NewName", "Status"]
    )

    # Detect duplicate proposed names and existing destinations
    if len(Plan):

        Valid = Plan.Status.eq("Rename") & Plan.NewName.notna()

        Key = Plan.Folder + "/" + Plan.NewName

        Twice = Key.duplicated(keep=False) & Valid

        Taken = Plan.apply(
            lambda R: (
                R.Status == "Rename"
                and (GroupDir / R.Folder / R.NewName).exists()
                and R.OldName != R.NewName
            ),
            axis=1
        )

        Plan.loc[Twice | Taken, "Status"] = "Conflict"

    return Plan


# Preview only
Apply = False
Undo = False

Plan = PlanRenames(Folders, Brackets=Brackets)

print(Plan.Status.value_counts().to_string(), "\n")
display(Plan)
print("PREVIEW ONLY — no files have been renamed.")


Status
Rename        17
Unreadable     9
Conflict       6
Check          2 



,Folder,OldName,NewName,Status
0,CalvingPenVideos,2026-09-26-CalvingPen1(1).avi,CalvingPen1Sep26Cow1.avi,Rename
1,CalvingPenVideos,2026-09-26-CalvingPen2(1).avi,CalvingPen2Sep26Cow1.avi,Rename
2,CalvingPenVideos,2026-09-26-CalvingPen2(2).avi,CalvingPen2Sep26Cow2.avi,Rename
3,CalvingPenVideos,2026-09-26-CalvingPen3(1).avi,CalvingPen3Sep26Cow1.avi,Conflict
4,CalvingPenVideos,2026-09-26-CalvingPen3(1)_ROI_01-06-51_to_01-26-29_2FPS.mp4.partial.mp4,CalvingPen3Sep26Cow1ROI.mp4,Conflict
5,CalvingPenVideos,2026-09-26-CalvingPen3(1)_ROI_01-42-00_to_01-57-00_2FPS.mp4,CalvingPen3Sep26Cow1ROI.mp4,Conflict
6,CalvingPenVideos,2026-09-26-CalvingPen3(2).avi,CalvingPen3Sep26Cow2.avi,Rename
7,CalvingPenVideos,2026-10-29 09-48-06 Calving Pen 2 (2).avi,CalvingPen2Oct29Cow2.avi,Unreadable
8,CalvingPenVideos,2026-10-29 09-48-06 Calving Pen 2 (3).avi,CalvingPen2Oct29Cow3.avi,Rename
9,CalvingPenVideos,2026-10-29 09-48-06 Calving Pen 2 (3)_ROI_01-06-51_to_01-26-29_2FPS.avi,CalvingPen2Oct29Cow3ROI.avi,Rename


PREVIEW ONLY — no files have been renamed.


## About this notebook

**Rule** (`CalvingName` in `CalvingPen/CalvingPipeline/Rename.py`): `CalvingPen<pen><Mon><DD>[Time<HHMMSS>]Part<n>[<rest in PascalCase>]` (no year; day with two digits, e.g. Nov01) (pure PascalCase: every number has a word in front, so numbers never run together). The part is the number in brackets after the pen (`Pen 3 (2)`) or after `Clip` (`Clip1Pen2`); without one it is `Part1`. Anything else in the old name (ROI, time window, fps) is kept at the end, so no information is lost.

**Status column:** `Rename` (will be renamed), `Same` (already right), `Check` (no date or pen in the name: left alone, rename by hand), `Unreadable` (broken Drive shortcut or still uploading: left alone), `Conflict` (the new name is already taken or two files would get it: left alone).

**Afterwards:** use the new names in `PipelineSteps` (`VideoFile`, and `VideoStem` = the new name without `.avi`). Old names still work for finding the video. Existing output folders keep their old names; only videos run from now on get new-style folder names.